# MediaPipe FaceMesh Inference on Full Test Set

This notebook runs **MediaPipe-only** eye-state inference on the entire test set and evaluates results with `classification_report()`.

Inference logic is taken from `implementation.py` by importing and using:
- `EyeAnalyzer`
- `SystemConfig`

No model logic is re-implemented in this notebook.

## 1) Import Dependencies and Implementation Functions

In [1]:
from pathlib import Path
import cv2
from sklearn.metrics import classification_report

# Import ONLY needed inference-related functions/classes from implementation.py
from implementation import EyeAnalyzer, SystemConfig

ModuleNotFoundError: No module named 'mediapipe'

## 2) Configure Test Set Paths (YOLO Format)

This cell supports two common YOLO-style layouts:
- Detection-style split with `images/` and `labels/`
- Classification-style split with class folders (e.g., `0 - close`, `1 - open`)

In [2]:
# Root test path
TEST_ROOT = Path("blinkblink-15/test")

# Optional YOLO detection-style directories
IMAGES_DIR = TEST_ROOT / "images"
LABELS_DIR = TEST_ROOT / "labels"

# Class-id mapping (dataset dependent)
CLASS_ID_TO_NAME = {
    0: "close",
    1: "open",
}
NAME_TO_CLASS_ID = {v: k for k, v in CLASS_ID_TO_NAME.items()}

# Common image extensions
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print(f"Test root exists: {TEST_ROOT.exists()} -> {TEST_ROOT}")
print(f"images/ exists: {IMAGES_DIR.exists()}")
print(f"labels/ exists: {LABELS_DIR.exists()}")

Test root exists: True -> blinkblink-15\test
images/ exists: False
labels/ exists: False


## 3) Load Image/Label Pairs from `blinkblink-15/test`

In [3]:
def collect_image_paths(test_root: Path):
    """Collect all candidate image files recursively, excluding labels directory."""
    image_paths = []
    for p in test_root.rglob("*"):
        if not p.is_file():
            continue
        if p.suffix.lower() not in IMAGE_EXTS:
            continue
        if "labels" in p.parts:
            continue
        image_paths.append(p)
    return sorted(image_paths)


def expected_label_path_for_image(img_path: Path, test_root: Path, labels_dir: Path):
    """Map image path to YOLO txt path when possible."""
    # Case A: images/<...>/img.ext -> labels/<...>/img.txt
    if "images" in img_path.parts:
        idx = img_path.parts.index("images")
        rel_after_images = Path(*img_path.parts[idx + 1:])
        return labels_dir / rel_after_images.with_suffix(".txt")

    # Case B: directly under test root with labels mirror structure
    rel = img_path.relative_to(test_root)
    return labels_dir / rel.with_suffix(".txt")


samples = []
image_paths = collect_image_paths(TEST_ROOT)

for img_path in image_paths:
    label_path = expected_label_path_for_image(img_path, TEST_ROOT, LABELS_DIR)
    samples.append(
        {
            "image_path": img_path,
            "label_path": label_path,
            "label_exists": label_path.exists(),
        }
    )

print(f"Total images found: {len(samples)}")
print(f"With YOLO txt labels: {sum(s['label_exists'] for s in samples)}")
print(f"Without YOLO txt labels: {sum(not s['label_exists'] for s in samples)}")

Total images found: 837
With YOLO txt labels: 0
Without YOLO txt labels: 837


## 4) Convert YOLO Labels to Eye-State Ground Truth

Deterministic conflict handling:
- If multiple class IDs exist in one label file, choose the **majority** class.
- If tie, choose the **smallest class ID**.
- If no txt label exists, fall back to class-folder name parsing (`close`/`open`).

In [4]:
def parse_yolo_class_ids(label_path: Path):
    class_ids = []
    if not label_path.exists():
        return class_ids

    with open(label_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            try:
                cid = int(float(parts[0]))
                class_ids.append(cid)
            except ValueError:
                continue
    return class_ids


def resolve_gt_class_id(sample):
    # Primary source: YOLO txt label file
    class_ids = parse_yolo_class_ids(sample["label_path"])
    if class_ids:
        counts = {}
        for cid in class_ids:
            counts[cid] = counts.get(cid, 0) + 1
        # Majority; tie -> smallest class id
        return sorted(counts.items(), key=lambda x: (-x[1], x[0]))[0][0], "yolo_label"

    # Fallback source: class-folder naming (YOLO classification-style export)
    parent_name = sample["image_path"].parent.name.lower()
    if "close" in parent_name or parent_name.startswith("0"):
        return 0, "folder_name"
    if "open" in parent_name or parent_name.startswith("1"):
        return 1, "folder_name"

    return None, "unresolved"


resolved_samples = []
unresolved = 0
for s in samples:
    gt_cid, gt_source = resolve_gt_class_id(s)
    row = dict(s)
    row["gt_class_id"] = gt_cid
    row["gt_source"] = gt_source
    resolved_samples.append(row)
    if gt_cid is None:
        unresolved += 1

print(f"Resolved ground truth: {len(resolved_samples) - unresolved}")
print(f"Unresolved ground truth: {unresolved}")

Resolved ground truth: 837
Unresolved ground truth: 0


## 5) Run FaceMesh Inference on the Full Test Set

Prediction rule (MediaPipe-only):
- `pred = open` if `normalized_ear >= config.blink_threshold`
- else `pred = close`

In [5]:
config = SystemConfig()
eye_analyzer = EyeAnalyzer(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)

inference_rows = []

for i, s in enumerate(resolved_samples, start=1):
    image_path = s["image_path"]
    gt_cid = s["gt_class_id"]

    # Skip unresolved GT rows
    if gt_cid is None:
        continue

    image = cv2.imread(str(image_path))
    if image is None:
        inference_rows.append(
            {
                "image_path": str(image_path),
                "gt_class_id": gt_cid,
                "pred_class_id": None,
                "status": "read_failed",
                "gt_source": s["gt_source"],
            }
        )
        continue

    eye_data, _ = eye_analyzer.process_frame(image, config)

    if not eye_data.landmarks_detected:
        inference_rows.append(
            {
                "image_path": str(image_path),
                "gt_class_id": gt_cid,
                "pred_class_id": None,
                "status": "no_face_landmarks",
                "gt_source": s["gt_source"],
            }
        )
        continue

    pred_cid = 1 if eye_data.normalized_ear >= config.blink_threshold else 0

    inference_rows.append(
        {
            "image_path": str(image_path),
            "gt_class_id": gt_cid,
            "pred_class_id": pred_cid,
            "status": "ok",
            "gt_source": s["gt_source"],
            "avg_ear": float(eye_data.avg_ear),
            "normalized_ear": float(eye_data.normalized_ear),
        }
    )

print(f"Inference rows: {len(inference_rows)}")
print(f"Successful predictions: {sum(r['status'] == 'ok' for r in inference_rows)}")
print(f"No-face rows: {sum(r['status'] == 'no_face_landmarks' for r in inference_rows)}")
print(f"Read-fail rows: {sum(r['status'] == 'read_failed' for r in inference_rows)}")

NameError: name 'SystemConfig' is not defined

## 6) Build Prediction and Target Arrays

In [6]:
valid_rows = [r for r in inference_rows if r["status"] == "ok" and r["pred_class_id"] is not None]

y_true = [r["gt_class_id"] for r in valid_rows]
y_pred = [r["pred_class_id"] for r in valid_rows]

print(f"Valid rows used for metrics: {len(valid_rows)}")
print(f"y_true length: {len(y_true)}")
print(f"y_pred length: {len(y_pred)}")

if len(y_true) != len(y_pred):
    raise ValueError("y_true and y_pred lengths do not match.")

NameError: name 'inference_rows' is not defined

## 7) Compute Classification Metrics with `classification_report()`

In [7]:
if len(y_true) == 0:
    print("No valid predictions available. Check no-face/read-fail counts above.")
else:
    report = classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["close", "open"],
        digits=4,
        zero_division=0,
    )
    print(report)

NameError: name 'y_true' is not defined

## 8) Display Per-Sample Results and Error Cases

In [ ]:
# Compact result table-like printout
preview_n = 20

print(f"Showing first {min(preview_n, len(valid_rows))} valid rows:")
for row in valid_rows[:preview_n]:
    match = row["gt_class_id"] == row["pred_class_id"]
    print({
        "path": row["image_path"],
        "y_true": CLASS_ID_TO_NAME.get(row["gt_class_id"], str(row["gt_class_id"])),
        "y_pred": CLASS_ID_TO_NAME.get(row["pred_class_id"], str(row["pred_class_id"])),
        "match": match,
    })

misclassified = [r for r in valid_rows if r["gt_class_id"] != r["pred_class_id"]]
print(f"\nMisclassified samples: {len(misclassified)}")
for row in misclassified[:50]:
    print(
        f"{row['image_path']} | "
        f"true={CLASS_ID_TO_NAME.get(row['gt_class_id'], row['gt_class_id'])} | "
        f"pred={CLASS_ID_TO_NAME.get(row['pred_class_id'], row['pred_class_id'])}"
    )

error_rows = [r for r in inference_rows if r["status"] != "ok"]
print(f"\nNon-evaluable rows: {len(error_rows)}")
for row in error_rows[:50]:
    print(f"{row['image_path']} | status={row['status']} | gt_source={row['gt_source']}")

In [ ]:
# Optional cleanup when done
eye_analyzer.close()